# Bài 2  · kNN và chuẩn hóa đặc trưng


# 0. Khởi tạo môi trường và nạp dữ liệu


In [8]:
from pathlib import Path
import sys
ROOT = Path("/content/MachineLearning/course/bai02")
if not (ROOT / 'source/sms_lab.py').exists():
    raise RuntimeError('Mở notebook tại thư mục Bai02 chứa source/ và data/.')
sys.path.insert(0, str(ROOT / 'source'))
from sms_lab import (load_data, FEATURES, np, pd, StandardScaler,
    KNeighborsClassifier, confusion_matrix, f1_score, threadpool_limits)
d, tr, va, te = load_data()
print('Sau bỏ trùng:', len(d), '; lớp:', d.label.value_counts().to_dict())
print('Train / validation / test:', len(tr), len(va), len(te))
print('Hai đặc trưng:', FEATURES)
for record in [2, 3]:
    row = d.loc[d.source_record == record].iloc[0]
    print(record, row.label, repr(row.text))
    print('Ký tự / chữ số / tỷ lệ:', row.n_chars, row.n_digits, row.digit_fraction)
assert d.loc[d.source_record == 3, 'n_chars'].iloc[0] == 155


Sau bỏ trùng: 5159 ; lớp: {'ham': 4517, 'spam': 642}
Train / validation / test: 3095 1032 1032
Hai đặc trưng: ['n_chars', 'digit_fraction']
2 ham 'Ok lar... Joking wif u oni...'
Ký tự / chữ số / tỷ lệ: 29 0 0.0
3 spam "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"
Ký tự / chữ số / tỷ lệ: 155 25 0.16129032258064516


## 1. Minh họa cách tính khoảng cách và bỏ phiếu

In [ ]:
P = np.array([[0,0], [2,0], [0,3], [4,4], [1,4]], dtype=float)
Y = np.array([0,1,1,0,0])  # 0 ham, 1 spam
query = np.array([1.,1.])
delta = P - query
d2 = (delta**2).sum(axis=1)
order = np.argsort(d2, kind='stable')
print('Chênh theo từng trục:\n', delta)
print('d²:', d2, '; d:', np.sqrt(d2))
for k in [3,5]:
    chosen = order[:k]
    spam_votes = int(Y[chosen].sum())
    prediction = int(spam_votes > k/2)
    print('k', k, '; top-k', chosen+1, '; spam/ham', spam_votes, k-spam_votes,
          '; dự đoán', ['ham','spam'][prediction])
assert np.array_equal(d2, [2,2,5,18,9])
assert set(order[:3]) == {0,1,2}


## 2. Trích xuất đặc trưng và chia dữ liệu


In [9]:
# Kiểm tra hai đặc trưng
print("Các đặc trưng:", FEATURES)

# Ktra kích thước các tập dữ liệu
print("Train:", len(tr))
print('Validation:', len(va))
print('Test:', len(te))

# Kiểm tra tỷ lệ nhãn trong từng tập
for name, indices in [('Train', tr), ('Validation', va), ('Test', te)]:
    print(f'\n{name}:')
    print(d.loc[indices, 'label'].value_counts())
    print(d.loc[indices, 'label'].value_counts(normalize=True).round(4))

Các đặc trưng: ['n_chars', 'digit_fraction']
Train: 3095
Validation: 1032
Test: 1032

Train:
label
ham     2710
spam     385
Name: count, dtype: int64
label
ham     0.8756
spam    0.1244
Name: proportion, dtype: float64

Validation:
label
ham     903
spam    129
Name: count, dtype: int64
label
ham     0.875
spam    0.125
Name: proportion, dtype: float64

Test:
label
ham     904
spam    128
Name: count, dtype: int64
label
ham     0.876
spam    0.124
Name: proportion, dtype: float64


In [ ]:
X_train = d.loc[tr, FEATURES].to_numpy()
y_train = d.loc[tr, 'y'].to_numpy()
scaler = StandardScaler().fit(X_train)
Z_train = scaler.transform(X_train)
print('μ:', scaler.mean_, '; σ:', scaler.scale_)
assert np.allclose(scaler.scale_, X_train.std(axis=0, ddof=0))
assert np.allclose(Z_train.mean(axis=0), 0, atol=1e-12)
assert np.allclose(Z_train.std(axis=0), 1, atol=1e-12)
tiny = np.array([80,80,120,120.])
print('Train mô phỏng: μ', tiny.mean(), '; σ', tiny.std(ddof=0))
print('z của 80,110,120:', (np.array([80,110,120])-100)/20)
changed_units = StandardScaler().fit_transform(X_train * [1,100])
assert np.allclose(changed_units, Z_train)


## 3.Tự cài đặt thuật toán kNN


In [ ]:
row = d.loc[d.source_record == 4500].iloc[0]
q = d.loc[d.source_record == 4500, FEATURES].to_numpy()
print('Text:', row.text)
print('x:', q[0], '; z:', scaler.transform(q)[0])
for scaled in [False,True]:
    T = Z_train if scaled else X_train
    Q = scaler.transform(q) if scaled else q
    squared = ((T-Q[0])**2).sum(axis=1)
    chosen = np.argsort(squared, kind='stable')[:3]
    model = KNeighborsClassifier(n_neighbors=3).fit(T, y_train)
    with threadpool_limits(limits=2):
        distance, ix = model.kneighbors(Q)
        prediction = int(model.predict(Q)[0])
    assert set(chosen) == set(ix[0])
    assert np.allclose(np.sort(squared[chosen]), np.sort(distance[0]**2))
    records = d.loc[tr[chosen], ['source_record','label',*FEATURES]].copy()
    records['distance'] = np.sqrt(squared[chosen])
    print('\nChuẩn hóa:', scaled)
    print(records.to_string(index=False))
    print('Dự đoán:', ['ham','spam'][prediction])
print('Nhãn để chấm sau dự đoán:', row.label)


## 4. Validation chọn cấu hình; tái lập test đã khóa
Cùng 3.095 train và 1.032 validation. F1 spam=2TP/(2TP+FP+FN), nhãn dương là spam. k11 được chọn trong ba k đã thử, theo F1. Test của  đã công bố:  tái lập snapshot, không xem như đánh giá mù mới, không tinh chỉnh bằng test.


In [ ]:
validation = []
models = {}
with threadpool_limits(limits=2):
    for scaled in [False,True]:
        for k in [3,5,11]:
            model = KNeighborsClassifier(n_neighbors=k).fit(
                Z_train if scaled else X_train, y_train)
            X_val = d.loc[va, FEATURES].to_numpy()
            pred = model.predict(scaler.transform(X_val) if scaled else X_val)
            tn, fp, fn, tp = map(int, confusion_matrix(d.loc[va,'y'],pred,labels=[0,1]).ravel())
            f1 = 2*tp/(2*tp+fp+fn)
            assert abs(f1-f1_score(d.loc[va,'y'],pred)) < 1e-12
            validation.append(dict(scaled=scaled,k=k,FP=fp,FN=fn,errors=fp+fn,F1=f1))
            models[(scaled,k)] = model
    best = max((x for x in validation if x['scaled']), key=lambda x:x['F1'])
    assert best['k'] == 11
    core = models[(True,11)]
    test_prediction = core.predict(scaler.transform(d.loc[te,FEATURES].to_numpy()))
cm = confusion_matrix(d.loc[te,'y'],test_prediction,labels=[0,1])
print(pd.DataFrame(validation).to_string(index=False))
print('Ma trận test: hàng thật, cột dự đoán; ham trước spam:\n',cm)
saved = pd.read_csv(ROOT/'data/sms_core_test_predictions.csv').set_index('message_id')
assert np.array_equal(test_prediction, saved.loc[d.loc[te,'message_id'],'prediction'])
assert np.array_equal(cm, [[892,12],[17,111]])


## 5. Một số điều kiện không được bỏ qua
k lẻ tránh hòa phiếu cho hai lớp, phiếu đều; chưa bảo đảm với nhiều lớp. Điểm hòa ở ranh giới k/k+1 khác nhãn có thể phụ thuộc thứ tự train. Cột hằng dùng scale=1 theo quy ước thư viện. Tỷ lệ phiếu chưa phải xác suất được hiệu chỉnh.


In [ ]:
constant = StandardScaler().fit([[5,1],[5,3]])
print('Cột hằng: scale', constant.scale_, '; z', constant.transform([[5,1],[5,3]]))
assert constant.scale_[0] == 1
q_units = np.array([100., .1])
AB = np.array([[101., .1],[100., .2]])
print('d² tỷ lệ:', ((AB-q_units)**2).sum(axis=1))
print('d² phần trăm:', (((AB-q_units)*[1,100])**2).sum(axis=1))
assert np.argmin(((AB-q_units)**2).sum(axis=1)) != np.argmin((((AB-q_units)*[1,100])**2).sum(axis=1))
votes_multiclass = np.bincount([0,0,1,1,2])
print('5 phiếu, ba lớp:', votes_multiclass, '→ hai lớp cùng 2 phiếu')


## 6. Vận dụng và tự kiểm
Tự tính query (2;1) trước khi chạy ô này. Rubric: tọa độ 2 điểm, khoảng cách/top3 3, phiếu/diễn giải 2, z/dùng lại train 3. Bài ở nhà: chọn SMS validation khác4500; truy vết text→x→z→top3 trước/sau; ghi dự đoán k11 và nhận xét có phạm vi.


In [ ]:
exit_d2 = ((P-[2,1])**2).sum(axis=1)
exit_neighbors = np.argsort(exit_d2,kind='stable')[:3]
print('Exit: d²',exit_d2,'; top3',exit_neighbors+1,'; phiếu spam',Y[exit_neighbors].sum())
assert np.array_equal(exit_d2,[5,1,8,13,10])
assert list(exit_neighbors+1) == [2,1,3]
assert (110-100)/20 == .5
from sms_lab import audit
receipt = audit()
print('Kiểm khoa học:',receipt['checks_count'],'/',len(receipt['checks']),';',receipt['status'])
print('Các phiên bản đã chạy:',receipt['versions'])


Nguồn: [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection), [StandardScaler 1.6](https://scikit-learn.org/1.6/modules/generated/sklearn.preprocessing.StandardScaler.html), [kNN 1.6](https://scikit-learn.org/1.6/modules/generated/sklearn.neighbors.KNeighborsClassifier.html). SHA dữ liệu nằm trong data/. Kiểm mã không thay việc đánh giá dự đoán ngoài train.
